In [1]:
#Imports

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (

    train_test_split,

    cross_val_score

)

from sklearn.metrics import (

    accuracy_score

)

from sklearn.ensemble import (

    StackingClassifier

)

from sklearn.linear_model import (

    LogisticRegression

)

from sklearn.tree import (

    DecisionTreeClassifier

)

from sklearn.neighbors import (

    KNeighborsClassifier

)

In [2]:
#Configurações do MLflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Stacking")

mlflow.sklearn.autolog()

2026/05/24 17:57:46 INFO mlflow.tracking.fluent: Experiment with name 'Stacking' does not exist. Creating a new experiment.


In [3]:
#Carregar o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [4]:
#Preparar os dados

df["YearsCodePro_Num"]=(

    df["YearsCodePro"]

)

df["YearsCodePro_Num"]=(

    df["YearsCodePro_Num"]

    .replace({

        "Less than 1 year":0,

        "More than 50 years":51,

        "Sem dado":0

    })

)

df["YearsCodePro_Num"]=pd.to_numeric(

    df["YearsCodePro_Num"]

)

features=[

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code"

]

target="JobSat_Class"

X=df[features]

y=df[target]

In [5]:
#Split dos dados

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

)

In [6]:
#Função do Stacking

def run_stacking_experiment(

    run_name,

    k

):

    with mlflow.start_run(

        run_name=run_name

    ):

        estimators=[

            (

                "lr",

                LogisticRegression()

            ),

            (

                "dt",

                DecisionTreeClassifier(

                    max_depth=3

                )

            ),

            (

                "knn",

                KNeighborsClassifier(

                    n_neighbors=k

                )

            )

        ]

        model=StackingClassifier(

            estimators=
            estimators,

            final_estimator=
            LogisticRegression()

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred=model.predict(

            X_test

        )

        accuracy=accuracy_score(

            y_test,

            y_pred

        )

        cv=cross_val_score(

            model,

            X_train,

            y_train,

            cv=5

        )

        mlflow.log_metric(

            "accuracy",

            accuracy

        )

        mlflow.log_metric(

            "cv_mean",

            cv.mean()

        )

        mlflow.log_metric(

            "cv_std",

            cv.std()

        )

        print(run_name)

        print(
            accuracy
        )

In [7]:
#Experiencia 0 - Baseline

run_stacking_experiment(

    "Experiment_0_Baseline",

    5

)

2026/05/24 17:59:24 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/05/24 17:59:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c

Experiment_0_Baseline
0.7493955512572534


In [8]:
#Experiencia 1 - K=7

run_stacking_experiment(

    "Experiment_1_K7",

    7

)

2026/05/24 18:05:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/24 18:05:31 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_1_K7
0.7486702127659575


In [9]:
#Experiencia 2 - K = 9

run_stacking_experiment(

    "Experiment_2_K9",

    9

)

2026/05/24 18:10:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/24 18:10:36 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_2_K9
0.7474310928433269


In [10]:
#Experiencia 3 - Adição de features

features_extra=[

    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5"

]

X=df[features_extra]

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

run_stacking_experiment(
    "Experiment_3_Features",
    5

)

2026/05/24 18:14:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarni

Experiment_3_Features
0.784634912959381


In [11]:
#Experiencia 4 - Random State

X_train,X_test,y_train,y_test=(

    train_test_split(

        X,

        y,

        test_size=0.2,

        random_state=7

    )

)

run_stacking_experiment(

    "Experiment_4_Random7",

    5

)

2026/05/24 18:31:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarni

Experiment_4_Random7
0.7930669729206963
